# Azure Pronunciation Assessment — APS Layer

**Author:**

Salvador Wahnon Palma s2665070@u.tsukuba.ac.jp

University of Tsukuba / Interaction Lab

<br>

**Objective:**

Automated Pronunciation Scoring (APS) layer for the feedback pipeline.
Sends a learner's audio to Azure Cognitive Services Pronunciation Assessment,
extracts phoneme-level accuracy scores, and returns mispronounced phonemes
in the format expected by `FeedbackGeneration.ipynb`.

This is a detection-only module — it does not connect to the LLM feedback generator.

---

## 1. Install & Imports

In [ ]:
# Run once to install the Azure Speech SDK
# !pip install azure-cognitiveservices-speech

In [1]:
import os
import json
import azure.cognitiveservices.speech as speechsdk

## 2. Azure Setup

Set these environment variables before running:

```
AZURE_SPEECH_KEY   — API key from the Azure portal
AZURE_SPEECH_REGION — e.g. "japaneast" or "eastasia"
```

Never hardcode credentials in this file.

In [2]:
SPEECH_KEY    = os.environ.get("AZURE_SPEECH_KEY")
SPEECH_REGION = os.environ.get("AZURE_SPEECH_REGION")

if not SPEECH_KEY or not SPEECH_REGION:
    raise EnvironmentError(
        "Set AZURE_SPEECH_KEY and AZURE_SPEECH_REGION environment variables before running."
    )

print(f"Azure region: {SPEECH_REGION}")
print("Key loaded:", "yes" if SPEECH_KEY else "no")

OSError: Set AZURE_SPEECH_KEY and AZURE_SPEECH_REGION environment variables before running.

## 3. Azure Phoneme → IPA Mapping

Azure returns phonemes in its own SAPI notation, not IPA.
This mapping converts Azure symbols to IPA so output is compatible with
the articulatory table and L1 knowledge base in `FeedbackGeneration.ipynb`.

**Known limitations:**
- `/N/` (moraic nasal): place of articulation is context-dependent (assimilates to following consonant). Mapped as-is; flagged in output.
- `/Q/` (geminate closure): duration is not captured by standard feature dimensions. Mapped as-is; flagged in output.
- Unrecognised symbols are passed through unchanged and logged for manual review.

In [ ]:
# Azure SAPI → IPA for ja-JP
# Expand as new phoneme symbols appear in Azure output logs.
AZURE_TO_IPA = {
    # Vowels
    "a":  "a",
    "i":  "i",
    "u":  "ɯ",
    "e":  "e",
    "o":  "o",
    # Stops
    "k":  "k",
    "g":  "ɡ",
    "t":  "t",
    "d":  "d",
    "b":  "b",
    "p":  "p",
    # Fricatives / affricates
    "s":  "s",
    "z":  "z",
    "sh": "ɕ",
    "ch": "tɕ",
    "ts": "ts",
    "h":  "h",
    "f":  "ɸ",
    # Nasals
    "m":  "m",
    "n":  "n",
    "N":  "N",   # moraic nasal — context-dependent place (methodology limitation)
    # Liquids / glides
    "r":  "ɾ",   # Japanese /r/ is a flap, not English /r/
    "y":  "j",
    "w":  "w",
    # Special
    "Q":  "Q",   # geminate closure — duration not in feature table (methodology limitation)
}

# Phonemes that need manual review / special handling
FLAGGED_PHONEMES = {"N", "Q"}


def azure_to_ipa(azure_phoneme):
    """Convert Azure SAPI phoneme symbol to IPA. Logs unrecognised symbols."""
    ipa = AZURE_TO_IPA.get(azure_phoneme)
    if ipa is None:
        print(f"  [LOG] Unrecognised Azure phoneme symbol: '{azure_phoneme}' — passing through for manual review")
        return azure_phoneme
    if ipa in FLAGGED_PHONEMES:
        print(f"  [LOG] Flagged phoneme /{ipa}/ — see methodology limitations in thesis")
    return ipa

## 4. Core Functions

In [ ]:
def assess_pronunciation(audio_path, reference_text, language="ja-JP"):
    """
    Send a WAV file to Azure Pronunciation Assessment.
    
    audio_path     — path to WAV file (16kHz, 16-bit, mono)
    reference_text — what the learner was supposed to say
    language       — BCP-47 locale tag (default: ja-JP)
    
    Returns the raw PronunciationAssessmentResult object.
    """
    speech_config = speechsdk.SpeechConfig(
        subscription=SPEECH_KEY,
        region=SPEECH_REGION
    )
    speech_config.speech_recognition_language = language

    audio_config = speechsdk.audio.AudioConfig(filename=audio_path)

    pronunciation_config = speechsdk.PronunciationAssessmentConfig(
        reference_text=reference_text,
        grading_system=speechsdk.PronunciationAssessmentGradingSystem.HundredMark,
        granularity=speechsdk.PronunciationAssessmentGranularity.Phoneme,
        enable_miscue=False
    )
    pronunciation_config.phoneme_level_timing_enabled = True

    recognizer = speechsdk.SpeechRecognizer(
        speech_config=speech_config,
        audio_config=audio_config
    )
    pronunciation_config.apply_to(recognizer)

    result = recognizer.recognize_once_async().get()

    if result.reason != speechsdk.ResultReason.RecognizedSpeech:
        raise RuntimeError(
            f"Recognition failed. Reason: {result.reason}\n"
            f"Cancellation details: {result.cancellation_details.error_details if result.cancellation_details else 'N/A'}"
        )

    return speechsdk.PronunciationAssessmentResult(result)

In [ ]:
def extract_mispronunciations(assessment_result, threshold=60):
    """
    Extract phonemes scored below threshold from a PronunciationAssessmentResult.
    
    Returns a list of dicts compatible with FeedbackGeneration.ipynb:
    [
        {
            "target_phoneme":  "/k:/",   # IPA notation
            "produced_phoneme": None,    # Azure rarely provides this; None when unavailable
            "accuracy_score":  23.4,
            "word":            "がっこう",
            "position":        2          # phoneme index within the word
        },
        ...
    ]
    
    When produced_phoneme is None, the feedback generator uses target phoneme
    features only (articulatory condition) or anchor word only (L1 condition).
    """
    mispronunciations = []

    for word in assessment_result.words:
        if not word.phonemes:
            continue
        for position, phoneme in enumerate(word.phonemes):
            if phoneme.accuracy_score < threshold:
                ipa = azure_to_ipa(phoneme.phoneme)
                mispronunciations.append({
                    "target_phoneme":   f"/{ipa}/",
                    "produced_phoneme": None,   # Azure PA does not return what was actually produced
                    "accuracy_score":   round(phoneme.accuracy_score, 1),
                    "word":             word.word,
                    "position":         position
                })

    return mispronunciations

## 5. Test Run

**Audio file requirement:** WAV, 16 kHz, 16-bit, mono.

Place your test file at the path below, then run this cell.
Record yourself or a non-native speaker saying the test word,
or generate a TTS version for initial smoke-testing.

Test words chosen for phonemes difficult for English speakers:
- `がっこう` (gakkou) — geminate /k:/ (moraic obstruent /Q/ + /k/)
- `さくら` (sakura) — flap /ɾ/ (not English /r/)

In [ ]:
# ── Test configuration ──────────────────────────────────────────────────────
TEST_AUDIO_PATH  = "test_audio/gakkou.wav"   # ← place your WAV file here
TEST_REFERENCE   = "がっこう"
THRESHOLD        = 60
# ────────────────────────────────────────────────────────────────────────────

if not os.path.isfile(TEST_AUDIO_PATH):
    print(f"[PLACEHOLDER] Audio file not found at '{TEST_AUDIO_PATH}'.")
    print("Create the test_audio/ folder and add a 16kHz/16-bit/mono WAV file to run this cell.")
else:
    print(f"=== Pronunciation Assessment: {TEST_REFERENCE} ===")
    print()

    raw_result = assess_pronunciation(TEST_AUDIO_PATH, TEST_REFERENCE)

    print(f"Overall accuracy : {raw_result.accuracy_score:.1f}")
    print(f"Fluency score    : {raw_result.fluency_score:.1f}")
    print(f"Completeness     : {raw_result.completeness_score:.1f}")
    print()

    mispronunciations = extract_mispronunciations(raw_result, threshold=THRESHOLD)

    if not mispronunciations:
        print(f"No phonemes scored below threshold ({THRESHOLD}).")
    else:
        print("Mispronounced phonemes:")
        for m in mispronunciations:
            print(
                f"  Word: {m['word']} | "
                f"Phoneme: {m['target_phoneme']} | "
                f"Accuracy: {m['accuracy_score']} | "
                f"Position: {m['position']}"
            )

        print()
        print("Ready for feedback pipeline:")
        for m in mispronunciations:
            print(f"  target_phoneme  : {m['target_phoneme']}")
            print(f"  produced_phoneme: {m['produced_phoneme']}")
            print(f"  accuracy_score  : {m['accuracy_score']}")
            print()

## 6. All-Phoneme Debug View

Prints every phoneme and its score — useful for calibrating the threshold
and auditing Azure's phoneme symbols before expanding `AZURE_TO_IPA`.

In [ ]:
if not os.path.isfile(TEST_AUDIO_PATH):
    print("[PLACEHOLDER] No audio file — skipping debug view.")
else:
    raw_result = assess_pronunciation(TEST_AUDIO_PATH, TEST_REFERENCE)

    print(f"All phonemes for '{TEST_REFERENCE}':")
    print(f"{'Word':<12} {'Pos':>3}  {'Azure':>6}  {'IPA':>6}  {'Score':>6}")
    print("-" * 42)
    for word in raw_result.words:
        if not word.phonemes:
            continue
        for pos, ph in enumerate(word.phonemes):
            ipa = AZURE_TO_IPA.get(ph.phoneme, f"?{ph.phoneme}")
            flag = " ← below threshold" if ph.accuracy_score < THRESHOLD else ""
            print(f"{word.word:<12} {pos:>3}  {ph.phoneme:>6}  {ipa:>6}  {ph.accuracy_score:>6.1f}{flag}")

---

## Known Limitations

| Issue | Effect | Mitigation |
|---|---|---|
| Azure does not return the actually-produced phoneme | `produced_phoneme` is always `None` | Articulatory condition uses target features only when this is `None` |
| `/N/` (moraic nasal) has context-dependent place | Feature table entry is underspecified | Flagged in output; noted as methodology limitation in thesis |
| `/Q/` (geminate closure) is a duration event, not a segment | Does not map to a standard articulatory feature row | Flagged in output; handled with fallback in feedback generator |
| Azure phoneme inventory may differ across SDK versions | Mapping may be incomplete | Unknown symbols are logged at runtime for manual review |